In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import numpy as np
import pickle as pkl
from openai import OpenAI
from sklearn.metrics import f1_score,roc_auc_score
import random
import json
from transformers import AutoTokenizer, AutoModelForCausalLM

## Load Dataset

In [5]:
city = 'weather_ny'

city_full_name = {
    'weather_ny': 'New York City'
}

In [10]:
# Load input data and ground truths
with open('./dataset/indices.pkl', 'rb') as f:
    indices = pkl.load(f)
    
# with open('dates.pkl', 'rb') as f:
#     dates = pkl.load(f)
    
with open(f'./dataset/time_series_{city}.pkl', 'rb') as f:
    data = pkl.load(f)

texts = {}
for i in indices:
    with open(os.path.join('./dataset/weather_summary', f'{city}_{i}.txt'), 'r') as f:
        text = f.read()
        texts[i] = text

gt_train = np.load('./dataset/gt_train.npy')
gt_val = np.load('./dataset/gt_val.npy')

# Load explanation results from the prototype-based encoder
with open('./expl_results/train_expl.json', 'r') as file:
    expl_train = json.load(file)

with open('./expl_results/val_expl.json', 'r') as file:
    expl_val = json.load(file)

with open('./expl_results/test_expl.json', 'r') as file:
    expl_test = json.load(file)

    


In [11]:
data_size = data.shape[0]
window_size = 24

data_size = len(indices)

num_train = int(data_size * 0.6)
num_test = int(data_size * 0.2)
num_vali = data_size - num_train - num_test

seq_len_day = 1

idx_train = np.arange(num_train - seq_len_day)
idx_val = np.arange(num_train - seq_len_day, num_train + num_vali - seq_len_day)
idx_test = np.arange(num_train + num_vali - seq_len_day, num_train + num_vali + num_test - seq_len_day)

## Prompt Qwen

In [17]:
local_model_path = "/Users/lqx/encoder/qwen-model"
tokenizer = AutoTokenizer.from_pretrained(local_model_path, trust_remote_code=True, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(local_model_path, trust_remote_code=True, local_files_only=True)
print("Model loaded.")

def generate_text(system_prompt, user_prompt, max_new_tokens=256, temperature=0.7):
    prompt = f"{system_prompt}\n{user_prompt}"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

Model loaded.


### Prediction - Text + Prototype 

In [ ]:
random.seed(2024)
llm_pred_tr0 = []
user_prompt_all = []
system_prompt = f"Your job is to act as a professional weather forecaster. You will be given a summary of the weather from the past 24 hours. Based on this information, your task is to predict whether it will rain in the next 24 hours."

k_max = 5 # 做多使用5个原型
for _i in idx_train:
    i = indices[_i]
    
    user_prompt = f"Your task is to predict whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. "

    prototypes =  expl_train['%d'%_i]
    
    k = k_max if len(prototypes) > k_max else len(prototypes)

    user_prompt += f'First, review the following {k_max} prototype text segments and outcomes, '
    user_prompt += 'so that you can refer to when making predictions:\n\n'

    for _k in range(k):
        
        user_prompt += f"Prototype #{_k+1}: {prototypes[_k]['Prototype']}"
        user_prompt += f"\nCorresponding Segment#{_k+1}: {prototypes[_k]['Input Segment']}"
        user_prompt += f"\nRelevance Score #{_k+1}: {np.round(prototypes[_k]['Similarity'],4)}"
        
        if prototypes[_k]['Class'] == 1:
            user_prompt += f"\nOutcome #{_k+1}: It rained.\n\n"
             
        else:
            user_prompt += f"\nOutcome #{_k+1}: It did not rain.\n\n"
 
    user_prompt += "Next, review the weather summary of the last 24 hours:\n\n"
    user_prompt += f"Summary: {texts[i]}\n\n"
    user_prompt += f"Outcome:\n\n"
    
    user_prompt += "Based on your understanding of the provided information of each prototype, "
    user_prompt += "predict the outcome of the current weather summary. "
    user_prompt += "Respond your prediction with either 'rain' or 'not rain'. "
    user_prompt += "Response should not include other terms."
    
    user_prompt_all.append(user_prompt)

    # Using Qwen
    text = generate_text(system_prompt, user_prompt, max_new_tokens=2048, temperature=0.3)

    llm_pred_tr0.append(text)

    
class_mapping = {'not rain':0, 'Not rain.':0,'rain': 1}
pllm_tr0 = [class_mapping[llm_pred_tr0[i]] for i in range(len(llm_pred_tr0))]

f1_mi_tr = f1_score(np.array(gt_train), np.array(pllm_tr0), average='micro')
f1_ma_tr = f1_score(np.array(gt_train), np.array(pllm_tr0), average='macro')
auc_tr = roc_auc_score(np.eye(2)[np.array(gt_train)],np.eye(2)[np.array(pllm_tr0)])

print(f1_mi_tr,f1_ma_tr,auc_tr)
 

### Reflection

In [ ]:
idx_incorrect01 = np.where((gt_train==0) & (np.array(pllm_tr0) == 1))[0]
idx_incorrect10 = np.where((gt_train==1) & (np.array(pllm_tr0) == 0))[0]

idx_correct0 = np.where((gt_train==0) & (np.array(pllm_tr0) == 0))[0]
idx_correct1 = np.where((gt_train==1) & (np.array(pllm_tr0) == 1))[0]


print(len(idx_incorrect01),len(idx_incorrect10),len(idx_correct0),len(idx_correct1))

 

In [ ]:
def sys_prompt_init(correct_flag):
    system_prompt = 'You are an advanced reasoning agent that can improve the quality of weather summary '
    system_prompt += 'based on self reflection. '
    system_prompt += 'You will be given the weather summaries, '
    system_prompt += f'and {correct_flag} predictions of whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. '
    system_prompt += 'Your task is to learn some reflections that guides the refinement of weather summaries.'
    return system_prompt

def sys_prompt_update(correct_flag):
    system_prompt = 'You are an advanced reasoning agent that can improve the quality of weather summary '
    system_prompt += 'based on self reflection. '
    system_prompt += 'You will receive a reflection report up to this point. '
    system_prompt += 'You will also be given new weather summaries, '
    system_prompt += f'and {correct_flag} predictions of whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. '
    system_prompt += 'Your task is to learn some reflections and update the current report that guides the refinement of weather summaries.'
    return system_prompt


 
def usr_prompt_init(correct_flag, idx, actual, pred, texts, gt_train, pllm_tr):
    user_prompt = f"Your task is to analyze the provided weather summaries with {correct_flag} predictions, so as"
    user_prompt += f" to generate a reflection report improving its quality for the prediction of whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. \n\n"
    user_prompt += f'Review the following {len(idx)} weather summaries with "{actual}" actual outcomes and "{pred}" predictions.\n'
    for _ii in range(len(idx)):
        i = idx[_ii]
        
        user_prompt += f"Summary #{_ii}: {texts[indices[i]]}\n"
        if gt_train[i] == 1:
            user_prompt += f"\nActual Outcome #{_ii}: It rained.\n"
        else:
            user_prompt += f"\nActual Outcome #{_ii}: It did not rain.\n"
        
        if pllm_tr[i] == 1:
            user_prompt += f"\nPrediction #{_ii}: It rained.\n\n"
        else:
            user_prompt += f"\nPrediction #{_ii}: It did not rain.\n\n"
            
 
    user_prompt += 'Reflection report: [Your Response]\n'
 
    user_prompt_shared = "Based on your analysis, write a high-quality reflection report that "
    if correct_flag == 'correct':
        user_prompt_shared += f'summarizes key phrases or sentences that led to correct predictions for "{actual}" outcomes. '
    else:
        user_prompt_shared += f'summarizes commonly misinterpreted and overlooked phrases or sentences that led to incorrectly predicting "{pred}" for "{actual}" actual outcomes. ' 
    user_prompt_shared += "Use precise terms to convey a clear and professional analysis, " 
    user_prompt_shared += "and avoid overly general statements. "
    user_prompt_shared += "The report should be a comprehensive and informative paragraph, "
    user_prompt_shared += "which can be generalized to refine similar weather summaries. "
    user_prompt_shared += "Your response should not include other terms." 
    
    return user_prompt + user_prompt_shared


def usr_prompt_update(correct_flag, idx, actual, pred, texts, gt_train, pllm_tr, refl):
    user_prompt = f"Your task is to analyze the provided weather summaries with {correct_flag} predictions, in order"
    user_prompt += f" to update the reflection report improving its quality for the prediction of whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. \n\n"
    user_prompt += 'First, review the following reflection report up to this point:\n'
    user_prompt += f'{refl}\n\n'
    
    user_prompt += f'Next, review the following {len(idx)} new weather summaries with "{actual}" actual outcomes and "{pred}" predictions.\n'
    for _ii in range(len(idx)):
        i = idx[_ii]
        
        user_prompt += f"Summary #{_ii}: {texts[indices[i]]}\n"
        if gt_train[i] == 1:
            user_prompt += f"\nActual Outcome #{_ii}: It rained.\n"
        else:
            user_prompt += f"\nActual Outcome #{_ii}: It did not rain.\n"
        
        if pllm_tr[i] == 1:
            user_prompt += f"\nPrediction #{_ii}: It rained.\n\n"
        else:
            user_prompt += f"\nPrediction #{_ii}: It did not rain.\n\n"
            
    user_prompt += 'Updated Reflection Report: [Your Response]\n'
 
    user_prompt_shared = "Based on your analysis, update the current reflection report so that it "
    if correct_flag == 'correct':
        user_prompt_shared += f'summarizes key phrases or sentences that led to correct predictions for "{actual}" outcomes. '
    else:
        user_prompt_shared += f'summarizes commonly misinterpreted and overlooked phrases or sentences that led to incorrectly predicting "{pred}" for "{actual}" actual outcomes. ' 
    user_prompt_shared += "Use precise terms to convey a clear and professional analysis, " 
    user_prompt_shared += "and avoid overly general statements. "
    user_prompt_shared += "The report should contain incremental and context-aware updates, "
    user_prompt_shared += "and can be generalized to refine similar weather summaries. "
    
     
    user_prompt_shared += "Your response should not include other terms." 
    
    return user_prompt + user_prompt_shared
 
    
 

correct_flags = ['correct','correct','incorrect','incorrect']
idxs = [idx_correct0, idx_correct1,idx_incorrect01,idx_incorrect10]
actuals = ['not rained','rained','not rained','rained']
preds =   ['not rained','rained','rained','not rained']
 
reflection_all = []
reflection_all_traj = []
batch_size = 20

for i in range(4):
    
    reflection_i_traj = []
    
    batches = [idxs[i][j:j+batch_size] for j in range(0, len(idxs[i]), batch_size)]
    for batch_id, batch in enumerate(batches):
        if batch_id == 0:
            system_prompt = sys_prompt_init(correct_flags[i])
            user_prompt = usr_prompt_init(correct_flags[i], batch, actuals[i], preds[i],
                        texts, gt_train, pllm_tr0)
        else:
            system_prompt = sys_prompt_update(correct_flags[i])
            user_prompt = usr_prompt_update(correct_flags[i], batch, actuals[i], preds[i],
                        texts, gt_train, pllm_tr0, refl)
    
        # Using Qwen
        text = generate_text(system_prompt, user_prompt, max_new_tokens=2048, temperature=0.3)
        reflection_i_traj.append(refl)
    reflection_all.append(refl)
    reflection_all_traj.append(reflection_i_traj)
        
        
 
 


In [ ]:
 
summarize_system_prompt = 'You are an advanced summarization agent that can generate high-quality summarization . '
summarize_system_prompt += 'You will be given previously generated reflections for text refinement, from the correct '
summarize_system_prompt += 'and incorrect predictions of weather texts. '
summarize_system_prompt += 'Your current task is to summarize these long reflections to better guide weather text refinement.'
 
    
summarize_user_prompt = f"Your task is to summarize the long reflections derived from previous predictions of weather contents. "
summarize_user_prompt += "The goal is to generate a high-quality report aimed at improving the weather text quality for better predictive accuracy. "

summarize_user_prompt += "First, review the reflections from all combinations of possible predictions and actual outcomes "
summarize_user_prompt += "that include 'not rained', and 'rained'.\n\n "

actuals = ['not rained','rained','not rained','rained']
preds =   ['not rained','rained','rained','not rained']
correct_flags = ['correct','correct','incorrect','incorrect']
for i in range(4):
    if correct_flags[i] == 'correct':
        msg = 'key phrases or sentences'
    else:
        msg = 'commonly misinterpreted and overlooked phrases or sentences'
    summarize_user_prompt += f"{i+1}. Review the following reflections that identify {msg} "
    summarize_user_prompt += f"for '{preds[i]}' predictions and '{actuals[i]}' actual outcomes.\n\n"
    summarize_user_prompt += f"{reflection_all[i]}\n\n"

 

summarize_user_prompt += "Based on your analysis, summarize the reflections of different scenarios and "
summarize_user_prompt += "write a comprehensive report that provides guidelines to select the most important "
summarize_user_prompt += "content in new weather texts where the actual outcome is unknown. "
summarize_user_prompt += "Your response should keep the enough details, yet effective, to improve the text quality for "
summarize_user_prompt += "downstream prediction. "
summarize_user_prompt += "Your response should not include other terms."

# Using Qwen
reflection_summary = generate_text(system_prompt, user_prompt, max_new_tokens=2048, temperature=0.3)

### Refinement

In [ ]:
 
refined_text = {}

In [ ]:

system_prompt = 'You are an advanced refinement agent designed to enhance the quality of weather summary. '
system_prompt += 'You will be provided with reflective thoughts analyzed from other weather summaries, '
system_prompt += 'and a weather summary that requires refinement. '
system_prompt += 'Your task is to generate a refined weather summary, '
system_prompt += 'by examining how reflective thoughts applied to the current summary.'
 
reflection = reflection_summary 


for _i in idx_train:
    i = _i
    print(i)
    user_prompt = f'Your task is to generate a refined weather summary from the current summary, '
    user_prompt += f'to improve its predictions of whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. '
    
    user_prompt += 'First, review the following reflections that provide guidelines for refinment:\n\n'
        
    user_prompt += f'{reflection}\n\n'
    user_prompt += 'Next, review the current weather summary that describes'
    user_prompt += 'the weather situation of the last 24 hours:\n\n'
    user_prompt += f'Summary: {texts[indices[i]]}\n\n'

    
    user_prompt += 'Based on your understanding, '
    user_prompt += 'generate a new weather summary by selecting relevant content in the current summary, '
    user_prompt += f'which provides insights crucial for understanding the weather situation in {city_full_name[city]} . '
    user_prompt += 'Response should not include other terms.'
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=[
        {
          "role": "system",
          "content": system_prompt
        },
        {
          "role": "user",
          "content": user_prompt
        }
        ],
        temperature=0.7,
        max_tokens=2048,
        top_p=1
    )

    refined_text[_i] = response.choices[0].message.content


In [ ]:
for _i in idx_val:
    i = _i
    print(i)
    user_prompt = f'Your task is to generate a refined weather summary from the current summary, '
    user_prompt += f'to improve its predictions of whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. '
    
    user_prompt += 'First, review the following reflections that provide guidelines for refinment:\n\n'
        
    user_prompt += f'{reflection}\n\n'
    user_prompt += 'Next, review the current weather summary that describes'
    user_prompt += 'the weather situation of the last 24 hours:\n\n'
    user_prompt += f'Summary: {texts[indices[i]]}\n\n'

    
    user_prompt += 'Based on your understanding, '
    user_prompt += 'generate a new weather summary by selecting relevant content in the current summary, '
    user_prompt += f'which provides insights crucial for understanding the weather situation in {city_full_name[city]} . '
    user_prompt += 'Response should not include other terms.'
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=[
        {
          "role": "system",
          "content": system_prompt
        },
        {
          "role": "user",
          "content": user_prompt
        }
        ],
        temperature=0.7,
        max_tokens=2048,
        top_p=1
    )
 
    refined_text[_i] = response.choices[0].message.content

#### Text Quality Comparison

In [ ]:
random.seed(2024)
llm_pred_va_s0 = []
user_prompt_all = []
system_prompt = f"Your job is to act as a professional weather forecaster. You will be given a summary of the weather from the past 24 hours. Based on this information, your task is to predict whether it will rain in the next 24 hours."

for _i in idx_val:
    i = indices[_i]
    
    user_prompt = f"Your task is to predict whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. "

 
    user_prompt += "First, review the weather summary of the last 24 hours:\n\n"
    user_prompt += f"Summary: {texts[i]}\n\n"
    user_prompt += f"Outcome:\n\n"
    
    user_prompt += "Based on your understanding, "
    user_prompt += "predict the outcome of the current weather summary. "
    user_prompt += "Respond your prediction with either 'rain' or 'not rain'. "
    user_prompt += "Response should not include other terms."
    
    user_prompt_all.append(user_prompt)
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=[
        {
          "role": "system",
          "content": system_prompt
        },
        {
          "role": "user",
          "content": user_prompt
        }
        ],
        temperature=0.3,
        max_tokens=2048,
        top_p=1
    )

    text = response.choices[0].message.content
    
    llm_pred_va_s0.append(text)

    
class_mapping = {'not rain': 0,'Not rain.': 0,'Not rain': 0, 'rain': 1, 'Rain': 1}
pllm_va_s0 = [class_mapping[llm_pred_va_s0[i]] for i in range(len(llm_pred_va_s0))]

f1_mi_va = f1_score(np.array(gt_val), np.array(pllm_va_s0), average='micro')
f1_ma_va = f1_score(np.array(gt_val), np.array(pllm_va_s0), average='macro')
auc_va = roc_auc_score(np.eye(2)[np.array(gt_val)],np.eye(2)[np.array(pllm_va_s0)])

print(f1_mi_va,f1_ma_va,auc_va)

In [ ]:
random.seed(2024)
llm_pred_va1 = []
user_prompt_all = []
system_prompt = f"Your job is to act as a professional weather forecaster. You will be given a summary of the weather from the past 24 hours. Based on this information, your task is to predict whether it will rain in the next 24 hours."

 
for _i in idx_val:
    
    user_prompt = f"Your task is to predict whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. "
 
    user_prompt += "First, review the weather summary of the last 24 hours:\n\n"
    user_prompt += f"Summary: {refined_text[_i]}\n\n"
    user_prompt += f"Outcome:\n\n"
    
    user_prompt += "Based on your understanding, "
    user_prompt += "predict the outcome of the current weather summary. "
    user_prompt += "Respond your prediction with either 'rain' or 'not rain'. "
    user_prompt += "Response should not include other terms."
    
    user_prompt_all.append(user_prompt)
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=[
        {
          "role": "system",
          "content": system_prompt
        },
        {
          "role": "user",
          "content": user_prompt
        }
        ],
        temperature=0.3,
        max_tokens=2048,
        top_p=1
    )

    text = response.choices[0].message.content
    
    llm_pred_va1.append(text)

    
class_mapping = {'not rain': 0,'Not rain.': 0,'Not rain': 0, 'rain': 1, 'Rain': 1}
 

pllm_va1 = [class_mapping[llm_pred_va1[i]] for i in range(len(llm_pred_va1))]

f1_mi_va1 = f1_score(np.array(gt_val), np.array(pllm_va1), average='micro')
f1_ma_va1 = f1_score(np.array(gt_val), np.array(pllm_va1), average='macro')
auc_va1 = roc_auc_score(np.eye(2)[np.array(gt_val)],np.eye(2)[np.array(pllm_va1)])

print(f1_mi_va1,f1_ma_va1,auc_va1)

#### Refinement on testing texts

In [ ]:

system_prompt = 'You are an advanced refinement agent designed to enhance the quality of weather summary. '
system_prompt += 'You will be provided with reflective thoughts analyzed from other weather summaries, '
system_prompt += 'and a weather summary that requires refinement. '
system_prompt += 'Your task is to generate a refined weather summary, '
system_prompt += 'by examining how reflective thoughts applied to the current summary.'

# To demonstrate the effectinvess of reflective refinement, we present the generated reflection along with the 
# corresponding refined testing texts of an iteration used in the iterative analysis of our paper. The final reflection
# is selected across multiple iterations and applied to the testing texts, as detailed in Algorithm 1.
reflection_best = reflection_summary 


for _i in idx_test:
    i = _i
    print(i)
    user_prompt = f'Your task is to generate a refined weather summary from the current summary, '
    user_prompt += f'to improve its predictions of whether it will rain or not in {city_full_name[city]} in the next {window_size} hours. '
    
    user_prompt += 'First, review the following reflections that provide guidelines for refinment:\n\n'
        
    user_prompt += f'{reflection_best}\n\n'
    user_prompt += 'Next, review the current weather summary that describes'
    user_prompt += 'the weather situation of the last 24 hours:\n\n'
    user_prompt += f'Summary: {texts[indices[i]]}\n\n'

    
    user_prompt += 'Based on your understanding, '
    user_prompt += 'generate a new weather summary by selecting relevant content in the current summary, '
    user_prompt += f'which provides insights crucial for understanding the weather situation in {city_full_name[city]} . '
    user_prompt += 'Response should not include other terms.'
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-08-06",
        messages=[
        {
          "role": "system",
          "content": system_prompt
        },
        {
          "role": "user",
          "content": user_prompt
        }
        ],
        temperature=0.7,
        max_tokens=2048,
        top_p=1
    )
 
    refined_text[_i] = response.choices[0].message.content

# The refined results are provided in ./dataset/refine_weather_summary.npy